In [ ]:
# SINGLE CONFIGURATION BLOCK — edit only this cell before a run.
RUN_MODE = "smoke"  # smoke or full
OUTPUT_DIR = "/content/counterfactual_faithfulness_stage1"
SEED = 17
MODEL_NAME = "dino_wm_pusht"
ENVIRONMENT = "PushT"
HORIZONS = [1, 2]  # world-model steps; each step is FRAMESKIP simulator actions
NUM_STATES = 10
ACTIONS_PER_STATE = 3

MOUNT_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/counterfactual_faithfulness_stage1"
REPO_URL = "https://github.com/facebookresearch/jepa-wms.git"
REPO_COMMIT = "13cf1d9c7e476f53c17714d2e0f1dc239a883ce0"
FRAMESKIP = 5
FEATURE_POOL_GRID = 4

if RUN_MODE == "full":
    NUM_STATES = 20
    ACTIONS_PER_STATE = 4
elif RUN_MODE != "smoke":
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

assert MODEL_NAME == "dino_wm_pusht"
assert ENVIRONMENT == "PushT"
assert 2 <= ACTIONS_PER_STATE <= 4
assert HORIZONS and min(HORIZONS) >= 1


# Stage 1: model and environment smoke test

This notebook validates one small public action-conditioned world model against
paired, executable Push-T interventions. It does **not** test a publication
hypothesis and it makes no claim about real-world robot reliability.

Run every cell in order from a fresh GPU runtime. A T4 with 16 GB is sufficient;
an L4 or A100 should be faster when available. The result ZIP automatically
downloads when execution finishes, including after a captured failure.
Google Drive is optional; set `MOUNT_DRIVE=True` above for resume files that
survive a runtime disconnect. No runtime restart is expected.


In [ ]:
import os
import subprocess
import sys

# Do not replace Colab's CUDA-matched torch/torchvision builds. All additional
# runtime dependencies are exact pins.
PINNED = [
    "einops==0.8.1",
    "tensordict==0.9.1",
    "timm==1.0.19",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "PyYAML==6.0.2",
    "huggingface_hub==0.36.2",
    "hf-xet==1.5.1",
    "gym==0.23.1",
    "pygame==2.6.1",
    "pymunk==6.8.0",
    "opencv-python-headless==4.11.0.86",
    "shapely==2.1.2",
    "lpips==0.1.4",
    "ruamel.yaml==0.18.10",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *PINNED],
    check=True,
)
print("Installed pinned non-PyTorch dependencies.")
print("No runtime restart is expected; continue to the next cell.")


In [ ]:
import contextlib
import csv
import hashlib
import json
import logging
import math
import platform
import random
import shutil
import subprocess
import sys
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import yaml

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_DIR

OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "intermediate").mkdir(exist_ok=True)
(OUT / "logs").mkdir(exist_ok=True)
(OUT / "plots").mkdir(exist_ok=True)

CACHE_ROOT = (
    Path("/content/drive/MyDrive/cf_faithfulness_cache")
    if MOUNT_DRIVE
    else Path("/content/cf_faithfulness_cache")
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["MPLCONFIGDIR"] = str(CACHE_ROOT / "matplotlib")
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["JEPAWM_LOGS"] = str(CACHE_ROOT / "unused_jepawm_logs")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

if not torch.cuda.is_available():
    raise RuntimeError("A Colab GPU runtime is required for the model smoke test.")

def version_tuple(text):
    return tuple(int(part) for part in text.split("+")[0].split(".")[:2])

if version_tuple(torch.__version__) < (2, 7):
    raise RuntimeError(f"JEPA-WMs requires torch>=2.7; Colab has {torch.__version__}")

VERSIONS = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory,
}
print(json.dumps(VERSIONS, indent=2))

LOG_PATH = OUT / "logs" / "run.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.FileHandler(LOG_PATH), logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("stage1")
log.info("Seeds set to %d", SEED)

def gpu_report(label):
    allocated = torch.cuda.memory_allocated() / 2**30
    reserved = torch.cuda.memory_reserved() / 2**30
    peak = torch.cuda.max_memory_allocated() / 2**30
    payload = {
        "label": label,
        "allocated_gib": round(allocated, 3),
        "reserved_gib": round(reserved, 3),
        "peak_allocated_gib": round(peak, 3),
    }
    log.info("GPU memory %s", payload)
    return payload

gpu_report("startup")


In [ ]:
# Metric, I/O, and simulator helpers are embedded so the notebook is standalone.
def pair_metrics(truth, prediction, eps=1e-12):
    # Canonical shape: state, alternative action, horizon, feature.
    truth = np.asarray(truth, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    if truth.shape != prediction.shape or truth.ndim != 4 or truth.shape[1] < 2:
        raise ValueError(f"invalid paired arrays: {truth.shape}, {prediction.shape}")
    errors = prediction - truth
    ordinary = np.sqrt(np.mean(errors**2, axis=(1, 3)))
    common = np.mean(errors, axis=1)
    common_rmse = np.sqrt(np.mean(common**2, axis=-1))
    centered = errors - common[:, None, :, :]
    action_dependent = np.sqrt(np.mean(centered**2, axis=(1, 3)))
    dy, dy_hat = [], []
    for left in range(truth.shape[1]):
        for right in range(left + 1, truth.shape[1]):
            dy.append(truth[:, left] - truth[:, right])
            dy_hat.append(prediction[:, left] - prediction[:, right])
    dy = np.stack(dy, axis=1)
    dy_hat = np.stack(dy_hat, axis=1)
    effect_error = dy_hat - dy
    effect_rmse = np.sqrt(np.mean(effect_error**2, axis=(1, 3)))
    effect_scale = np.sqrt(np.mean(dy**2, axis=(1, 3)))
    normalized = effect_rmse / np.maximum(effect_scale, eps)
    dot = np.sum(dy_hat * dy, axis=-1)
    denom = np.linalg.norm(dy_hat, axis=-1) * np.linalg.norm(dy, axis=-1)
    cosine = np.divide(
        dot, denom, out=np.full_like(dot, np.nan), where=denom > eps
    )
    cosine = np.nanmean(cosine, axis=1)
    pair_mse = effect_rmse**2
    expected = (
        2 * truth.shape[1] / (truth.shape[1] - 1)
    ) * action_dependent**2
    return {
        "ordinary_rmse": ordinary,
        "common_mode_rmse": common_rmse,
        "action_dependent_rmse": action_dependent,
        "paired_effect_rmse": effect_rmse,
        "ground_truth_effect_rms": effect_scale,
        "normalized_paired_effect_rmse": normalized,
        "paired_effect_cosine": cosine,
        "identity_residual": pair_mse - expected,
    }

def rank_metrics(true_cost, predicted_cost, eps=1e-12):
    true_cost = np.asarray(true_cost, dtype=np.float64)
    predicted_cost = np.asarray(predicted_cost, dtype=np.float64)
    selected = np.argmin(predicted_cost, axis=1)
    oracle = np.argmin(true_cost, axis=1)
    rows, horizons = np.indices(selected.shape)
    chosen = true_cost[rows, selected, horizons]
    best = np.min(true_cost, axis=1)
    regret = chosen - best
    spread = np.max(true_cost, axis=1) - best
    norm_regret = np.divide(
        regret, np.maximum(spread, eps), out=np.zeros_like(regret), where=spread > eps
    )
    concordant = np.zeros_like(regret)
    compared = np.zeros_like(regret)
    for left in range(true_cost.shape[1]):
        for right in range(left + 1, true_cost.shape[1]):
            dt = true_cost[:, left] - true_cost[:, right]
            dp = predicted_cost[:, left] - predicted_cost[:, right]
            valid = np.abs(dt) > 1e-9
            compared += valid
            concordant += valid & (np.sign(dt) == np.sign(dp))
    pairwise = np.divide(
        concordant, compared, out=np.full_like(regret, np.nan), where=compared > 0
    )
    return {
        "selected_action": selected,
        "oracle_action": oracle,
        "top1_correct": (chosen <= best + 1e-9).astype(float),
        "regret": regret,
        "normalized_regret": norm_regret,
        "pairwise_accuracy": pairwise,
    }

def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        raise ValueError(f"no rows for {path}")
    with open(path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value

def write_json(path, payload):
    Path(path).write_text(json.dumps(json_ready(payload), indent=2) + "\n")

def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()

def task_cost(state, goal=np.array([256.0, 256.0, np.pi / 4])):
    state = np.asarray(state)
    angle = math.atan2(math.sin(state[4] - goal[2]), math.cos(state[4] - goal[2]))
    return float(np.linalg.norm(np.r_[(state[2:4] - goal[:2]) / 512.0, angle / np.pi]))

def pool_visual(visual, grid=FEATURE_POOL_GRID):
    # Accept [..., view, H, W, D] and retain a coarse spatial grid.
    value = visual.detach().float().cpu().numpy()[..., 0, :, :, :]
    h, w, dim = value.shape[-3:]
    if h % grid or w % grid:
        raise ValueError(f"feature grid {(h, w)} is not divisible by {grid}")
    fh, fw = h // grid, w // grid
    value = value.reshape(*value.shape[:-3], grid, fh, grid, fw, dim)
    value = value.mean(axis=(-4, -2))
    return value.reshape(*value.shape[:-3], -1)

def build_states(count):
    rng = np.random.default_rng(SEED)
    states = []
    for idx in range(count):
        block_x = 256.0 + rng.uniform(-22.0, 22.0)
        block_y = 300.0 + rng.uniform(-12.0, 12.0)
        angle = rng.uniform(-0.22, 0.22)
        agent_x = block_x + rng.uniform(-12.0, 12.0)
        agent_y = block_y + 120.0 + rng.uniform(-4.0, 4.0)
        states.append([agent_x, agent_y, block_x, block_y, angle, 0.0, 0.0])
    return np.asarray(states, dtype=np.float64)

def build_action_bank(count, primitive_steps):
    patterns = [
        np.array([0.0, 0.0]),
        np.array([0.0, -0.22]),
        np.array([-0.18, -0.16]),
        np.array([0.18, -0.16]),
    ]
    return np.stack([np.tile(pattern, (primitive_steps, 1)) for pattern in patterns[:count]])

def configure_repo():
    repo = CACHE_ROOT / "jepa-wms"
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", REPO_COMMIT], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REPO_COMMIT], check=True)
    resolved = subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
    ).strip()
    if resolved != REPO_COMMIT:
        raise RuntimeError(f"repository pin mismatch: {resolved}")

    # The official torch-hub loader imports the entire planner stack even though
    # this notebook only needs model initialization. Patch that import at the
    # pinned revision to reduce dependencies; keep model code and weights intact.
    hubconf = repo / "hubconf.py"
    text = hubconf.read_text()
    old = "from evals.simu_env_planning.eval import init_module"
    new = (
        "from app.vjepa_wm.modelcustom.simu_env_planning."
        "vit_enc_preds import init_module"
    )
    if old in text:
        hubconf.write_text(text.replace(old, new))
    elif new not in text:
        raise RuntimeError("unexpected hubconf.py at pinned commit")

    # Avoid importing DROID video dependencies for an unused helper.
    model_file = (
        repo / "app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py"
    )
    text = model_file.read_text()
    old = "from app.plan_common.datasets.droid_dset import compute_new_pose"
    new = (
        "def compute_new_pose(*args, **kwargs):\n"
        "    raise RuntimeError('compute_new_pose is unused in Push-T predict_proprio mode')"
    )
    if old in text:
        model_file.write_text(text.replace(old, new))
    elif "compute_new_pose is unused" not in text:
        raise RuntimeError("unexpected vit_enc_preds.py at pinned commit")

    # Decoder heads are unnecessary for latent metrics and the optional image
    # decoder is 3.64 GB. Disable heads in the copied evaluation config only.
    config_path = (
        repo
        / "configs/evals/simu_env_planning/pt/dino-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml"
    )
    config = yaml.safe_load(config_path.read_text())
    heads = config["model_kwargs"]["pretrain_kwargs"]["heads_cfg"]
    heads["architectures"] = {}
    heads["pretrain_dec_path"] = None
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    return repo

def make_environment(repo):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from evals.simu_env_planning.envs.pusht_env.pusht_env import PushTEnv
    return PushTEnv(
        with_velocity=True,
        with_target=True,
        render_size=224,
        relative=True,
        action_scale=100,
    )

def reset_env(env, state, seed):
    env.seed(seed)
    env.reset_to_state = np.asarray(state, dtype=np.float64).copy()
    observation, restored_state = env.reset()
    return {
        "visual": np.asarray(observation["visual"]).copy(),
        "proprio": np.asarray(observation["proprio"]).copy(),
    }, np.asarray(restored_state).copy()

def rollout_branch(env, state, primitive_actions, horizons, seed):
    observation0, restored = reset_env(env, state, seed)
    wanted = set(horizons)
    horizon_observations = {}
    horizon_states = {}
    horizon_contacts = {}
    cumulative_contacts = 0
    for step, action in enumerate(primitive_actions, start=1):
        observation, _, _, info = env.step(action)
        cumulative_contacts += int(info.get("n_contacts", 0))
        if step % FRAMESKIP == 0:
            model_horizon = step // FRAMESKIP
            if model_horizon in wanted:
                horizon_observations[model_horizon] = {
                    "visual": np.asarray(observation["visual"]).copy(),
                    "proprio": np.asarray(observation["proprio"]).copy(),
                }
                horizon_states[model_horizon] = np.asarray(info["state"]).copy()
                horizon_contacts[model_horizon] = cumulative_contacts
    missing = wanted - set(horizon_observations)
    if missing:
        raise RuntimeError(f"missing simulator horizons {missing}")
    return observation0, restored, horizon_observations, horizon_states, horizon_contacts

def to_model_observation(observation):
    visual = torch.from_numpy(observation["visual"]).permute(2, 0, 1)
    proprio = torch.from_numpy(observation["proprio"])
    return {
        "visual": visual.unsqueeze(0).unsqueeze(0),
        "proprio": proprio.unsqueeze(0).unsqueeze(0),
    }

def exact_restore_test(env, state, actions, repeats=3):
    endpoints, initial_images, contacts = [], [], []
    for _ in range(repeats):
        obs0, _, _, states, contact = rollout_branch(
            env, state, actions, [max(HORIZONS)], SEED + 1000
        )
        endpoints.append(states[max(HORIZONS)])
        initial_images.append(obs0["visual"])
        contacts.append(contact[max(HORIZONS)])
    endpoint_exact = all(np.array_equal(endpoints[0], value) for value in endpoints[1:])
    image_exact = all(np.array_equal(initial_images[0], value) for value in initial_images[1:])
    diagnostics_exact = len(set(contacts)) == 1
    if not (endpoint_exact and image_exact and diagnostics_exact):
        raise AssertionError("restored Push-T branches are not bitwise identical")
    return {
        "repeats": repeats,
        "endpoint_bitwise_exact": endpoint_exact,
        "initial_render_bitwise_exact": image_exact,
        "diagnostics_exact": diagnostics_exact,
        "max_endpoint_abs_diff": float(
            max(np.max(np.abs(endpoints[0] - value)) for value in endpoints[1:])
        ),
    }


In [ ]:
def execute_stage1():
    config_payload = {
        "RUN_MODE": RUN_MODE,
        "OUTPUT_DIR": OUTPUT_DIR,
        "SEED": SEED,
        "MODEL_NAME": MODEL_NAME,
        "ENVIRONMENT": ENVIRONMENT,
        "HORIZONS": HORIZONS,
        "NUM_STATES": NUM_STATES,
        "ACTIONS_PER_STATE": ACTIONS_PER_STATE,
        "MOUNT_DRIVE": MOUNT_DRIVE,
        "REPO_URL": REPO_URL,
        "REPO_COMMIT": REPO_COMMIT,
        "FRAMESKIP": FRAMESKIP,
        "FEATURE_POOL_GRID": FEATURE_POOL_GRID,
        "pinned_dependencies": PINNED,
    }
    signature = hashlib.sha256(
        json.dumps(config_payload, sort_keys=True).encode()
    ).hexdigest()
    write_json(OUT / "config.json", {**config_payload, "run_signature": signature})
    write_json(OUT / "versions.json", VERSIONS)

    progress_path = OUT / "progress.json"
    if progress_path.exists():
        previous = json.loads(progress_path.read_text())
        if previous.get("run_signature") != signature:
            raise RuntimeError(
                "OUTPUT_DIR contains checkpoints from a different configuration; "
                "choose a new OUTPUT_DIR."
            )

    repo = configure_repo()
    log.info("Pinned JEPA-WMs repository at %s", REPO_COMMIT)
    env = make_environment(repo)
    states = build_states(NUM_STATES)
    max_primitive_steps = max(HORIZONS) * FRAMESKIP
    action_bank = build_action_bank(ACTIONS_PER_STATE, max_primitive_steps)

    restore = exact_restore_test(env, states[0], action_bank[1])
    write_json(OUT / "restore_test.json", restore)
    log.info("Exact restoration test: %s", restore)

    import pymunk
    environment_payload = {
        "name": "JEPA-WMs bundled PushTEnv",
        "repository": REPO_URL,
        "commit": REPO_COMMIT,
        "pymunk": pymunk.version,
        "relative_actions": True,
        "action_scale": 100,
        "with_velocity": True,
        "frameskip": FRAMESKIP,
        "branch_protocol": "fresh space, stationary canonical state",
    }
    write_json(OUT / "environment.json", environment_payload)

    gpu_report("before_model_load")
    model, preprocessor = torch.hub.load(
        str(repo),
        MODEL_NAME,
        source="local",
        pretrained=True,
        device="cuda:0",
        trust_repo=True,
    )
    model.eval()
    gpu_report("after_model_load")

    # Fixed Push-T goal observation for the DINO latent planning cost.
    goal_state = np.array([80.0, 450.0, 256.0, 256.0, np.pi / 4, 0.0, 0.0])
    goal_observation, _ = reset_env(env, goal_state, SEED + 2000)
    with torch.inference_mode():
        goal_encoded = model.encode(to_model_observation(goal_observation))
    goal_feature = pool_visual(goal_encoded["visual"])[0, 0]

    for state_idx, state in enumerate(states):
        state_path = OUT / "intermediate" / f"state_{state_idx:03d}.npz"
        if state_path.exists():
            log.info("Resume: keeping completed %s", state_path.name)
            continue

        # Identical initial observation for every alternative.
        initial_observation, _ = reset_env(env, state, SEED + state_idx)
        with torch.inference_mode():
            z0 = model.encode(to_model_observation(initial_observation))

        truth_by_action, prediction_by_action = [], []
        true_cost_by_action, predicted_cost_by_action = [], []
        contact_by_action, endpoint_state_by_action = [], []

        for action_idx, primitive_actions in enumerate(action_bank):
            _, _, horizon_obs, horizon_states, horizon_contacts = rollout_branch(
                env,
                state,
                primitive_actions,
                HORIZONS,
                SEED + state_idx,
            )
            future_visual = torch.stack(
                [
                    torch.from_numpy(horizon_obs[h]["visual"]).permute(2, 0, 1)
                    for h in HORIZONS
                ]
            ).unsqueeze(0)
            future_proprio = torch.stack(
                [torch.from_numpy(horizon_obs[h]["proprio"]) for h in HORIZONS]
            ).unsqueeze(0)

            chunks = torch.from_numpy(
                primitive_actions.reshape(max(HORIZONS), FRAMESKIP, 2)
            ).float()
            normalized_chunks = preprocessor.normalize_actions(chunks)
            model_actions = normalized_chunks.reshape(max(HORIZONS), 1, -1).cuda()

            with torch.inference_mode():
                z_truth = model.encode(
                    {"visual": future_visual, "proprio": future_proprio}
                )
                z_prediction = model.unroll(z0, model_actions)

            truth_features = pool_visual(z_truth["visual"])[0]
            predicted_visual = z_prediction["visual"]
            prediction_features = np.stack(
                [pool_visual(predicted_visual[h : h + 1])[0, 0] for h in HORIZONS]
            )
            truth_by_action.append(truth_features)
            prediction_by_action.append(prediction_features)
            true_cost_by_action.append(
                [task_cost(horizon_states[h]) for h in HORIZONS]
            )
            predicted_cost_by_action.append(
                [
                    float(np.sqrt(np.mean((prediction_features[pos] - goal_feature) ** 2)))
                    for pos in range(len(HORIZONS))
                ]
            )
            contact_by_action.append([horizon_contacts[h] for h in HORIZONS])
            endpoint_state_by_action.append([horizon_states[h] for h in HORIZONS])

        truth_array = np.stack(truth_by_action)
        prediction_array = np.stack(prediction_by_action)
        true_cost_array = np.asarray(true_cost_by_action)
        predicted_cost_array = np.asarray(predicted_cost_by_action)
        contact_array = np.asarray(contact_by_action)
        endpoint_state_array = np.asarray(endpoint_state_by_action)

        # Action, horizon, feature. The state axis is added after resume loading.
        np.savez_compressed(
            state_path,
            truth=truth_array,
            prediction=prediction_array,
            true_cost=true_cost_array,
            predicted_cost=predicted_cost_array,
            contacts=contact_array,
            endpoint_states=endpoint_state_array,
            initial_state=state,
        )
        write_json(
            progress_path,
            {
                "run_signature": signature,
                "completed_states": state_idx + 1,
                "total_states": NUM_STATES,
                "last_file": state_path.name,
            },
        )
        log.info("Completed state %d/%d", state_idx + 1, NUM_STATES)
        gpu_report(f"state_{state_idx:03d}")

    loaded = [
        np.load(OUT / "intermediate" / f"state_{idx:03d}.npz")
        for idx in range(NUM_STATES)
    ]
    truth = np.stack([item["truth"] for item in loaded])
    prediction = np.stack([item["prediction"] for item in loaded])
    true_cost = np.stack([item["true_cost"] for item in loaded])
    predicted_cost = np.stack([item["predicted_cost"] for item in loaded])
    contacts = np.stack([item["contacts"] for item in loaded])
    endpoint_states = np.stack([item["endpoint_states"] for item in loaded])
    for item in loaded:
        item.close()

    paired = pair_metrics(truth, prediction)
    ranking = rank_metrics(true_cost, predicted_cost)
    distinct_effects = np.max(
        np.linalg.norm(
            endpoint_states[:, :, None, :, :] - endpoint_states[:, None, :, :, :],
            axis=-1,
        )
    )
    if distinct_effects <= 1e-9:
        raise AssertionError("alternative actions produced no distinct simulator outcomes")
    if np.max(np.abs(paired["identity_residual"])) > 1e-8:
        raise AssertionError("paired metric identity check failed")
    if not all(np.all(np.isfinite(value)) for key, value in paired.items() if key != "paired_effect_cosine"):
        raise AssertionError("non-finite paired metrics")

    unit_rows, ranking_rows = [], []
    for state_idx in range(NUM_STATES):
        for horizon_pos, horizon in enumerate(HORIZONS):
            contact_any = bool(np.any(contacts[state_idx, :, horizon_pos] > 0))
            unit_rows.append(
                {
                    "state_id": state_idx,
                    "model": MODEL_NAME,
                    "horizon": horizon,
                    "contact_any": int(contact_any),
                    **{
                        key: float(value[state_idx, horizon_pos])
                        for key, value in paired.items()
                    },
                }
            )
            ranking_rows.append(
                {
                    "state_id": state_idx,
                    "model": MODEL_NAME,
                    "horizon": horizon,
                    "contact_any": int(contact_any),
                    **{
                        key: (
                            int(value[state_idx, horizon_pos])
                            if key in {"selected_action", "oracle_action"}
                            else float(value[state_idx, horizon_pos])
                        )
                        for key, value in ranking.items()
                    },
                }
            )
    write_csv(OUT / "unit_metrics.csv", unit_rows)
    write_csv(OUT / "ranking_metrics.csv", ranking_rows)

    summary_rows = []
    for horizon_pos, horizon in enumerate(HORIZONS):
        summary_rows.append(
            {
                "model": MODEL_NAME,
                "horizon": horizon,
                "num_states": NUM_STATES,
                "actions_per_state": ACTIONS_PER_STATE,
                **{
                    key: float(np.nanmean(value[:, horizon_pos]))
                    for key, value in paired.items()
                },
                "top1_accuracy": float(
                    np.mean(ranking["top1_correct"][:, horizon_pos])
                ),
                "mean_regret": float(np.mean(ranking["regret"][:, horizon_pos])),
                "mean_normalized_regret": float(
                    np.mean(ranking["normalized_regret"][:, horizon_pos])
                ),
                "mean_pairwise_accuracy": float(
                    np.nanmean(ranking["pairwise_accuracy"][:, horizon_pos])
                ),
            }
        )
    write_csv(OUT / "metrics_summary.csv", summary_rows)
    write_json(
        OUT / "metrics_summary.json",
        {"status": "SUCCESS", "rows": summary_rows, "pipeline_only": True},
    )
    np.savez_compressed(
        OUT / "all_results.npz",
        truth=truth,
        prediction=prediction,
        true_cost=true_cost,
        predicted_cost=predicted_cost,
        contacts=contacts,
        endpoint_states=endpoint_states,
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(
        paired["ordinary_rmse"].ravel(),
        paired["paired_effect_rmse"].ravel(),
        c=ranking["normalized_regret"].ravel(),
        cmap="viridis",
    )
    axes[0].set_xlabel("ordinary latent RMSE")
    axes[0].set_ylabel("paired effect RMSE")
    axes[0].set_title("Each point is one state × horizon")
    axes[1].plot(
        HORIZONS,
        np.nanmean(paired["normalized_paired_effect_rmse"], axis=0),
        marker="o",
        label="normalized paired error",
    )
    axes[1].plot(
        HORIZONS,
        np.mean(ranking["normalized_regret"], axis=0),
        marker="s",
        label="normalized regret",
    )
    axes[1].set_xlabel("world-model horizon")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(OUT / "plots" / "stage1_metrics.png", dpi=180)
    plt.close(fig)

    # Record actual cached model artifacts rather than assuming downloads.
    candidates = []
    for root in [Path(os.environ["HF_HOME"]), Path(os.environ["TORCH_HOME"])]:
        if root.exists():
            for path in root.rglob("*"):
                if path.is_file() and (
                    "dino_wm_pusht" in path.name
                    or "dinov2_vits14" in path.name
                    or "dinov2_vits14" in str(path)
                ):
                    candidates.append(path)
    manifest = []
    for path in sorted(set(candidates)):
        manifest.append(
            {
                "path": str(path),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
    write_json(
        OUT / "checkpoints_manifest.json",
        {
            "model": MODEL_NAME,
            "expected_checkpoint": "facebook/jepa-wms:dino_wm_pusht.pth.tar",
            "expected_encoder": "facebookresearch/dinov2:dinov2_vits14",
            "dataset_downloaded": False,
            "cached_files": manifest,
        },
    )
    (OUT / "FAILURE_TRACE.txt").write_text("NONE\n")
    gpu_report("complete")
    return summary_rows

def package_results():
    archive_base = OUT.parent / "stage1_result_bundle"
    archive_path = Path(
        shutil.make_archive(str(archive_base), "zip", root_dir=OUT)
    )
    print(f"RESULT ZIP: {archive_path}")
    return archive_path

try:
    SUMMARY = execute_stage1()
    RUN_STATUS = "SUCCESS"
except Exception:
    RUN_STATUS = "FAILED"
    failure = traceback.format_exc()
    (OUT / "FAILURE_TRACE.txt").write_text(failure)
    log.exception("Stage 1 failed")
finally:
    RESULT_ZIP = package_results()
    try:
        from google.colab import files
        files.download(str(RESULT_ZIP))
        print(f"Automatic download requested: {RESULT_ZIP.name}")
    except Exception as download_exc:
        print("Automatic download unavailable; use the Colab Files pane.", download_exc)

print("RUN_STATUS:", RUN_STATUS)
if RUN_STATUS != "SUCCESS":
    raise RuntimeError(
        f"Stage 1 failed. Download {RESULT_ZIP} so FAILURE_TRACE.txt can be inspected."
    )


In [ ]:
# Final success-only sanity checks. The preceding cell already requested the download.
required = [
    "config.json",
    "versions.json",
    "environment.json",
    "restore_test.json",
    "checkpoints_manifest.json",
    "unit_metrics.csv",
    "ranking_metrics.csv",
    "metrics_summary.csv",
    "metrics_summary.json",
    "all_results.npz",
    "FAILURE_TRACE.txt",
    "logs/run.log",
    "plots/stage1_metrics.png",
]
missing = [name for name in required if not (OUT / name).exists()]
if missing:
    raise AssertionError(f"result bundle is missing: {missing}")
restore = json.loads((OUT / "restore_test.json").read_text())
summary = json.loads((OUT / "metrics_summary.json").read_text())
assert restore["endpoint_bitwise_exact"]
assert restore["initial_render_bitwise_exact"]
assert summary["status"] == "SUCCESS"
assert (OUT / "FAILURE_TRACE.txt").read_text().strip() == "NONE"

print(json.dumps(summary, indent=2))
print(f"Sanity checks passed. Return this file: {RESULT_ZIP.name}")
print(f"Path: {RESULT_ZIP}")
print("The automatic browser download was requested by the execution cell.")
